# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**URL:** [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

In [ ]:
# List all available record sets along with their '@id' fields
record_sets = dataset.metadata['recordSet'] if 'recordSet' in dataset.metadata else []
if not record_sets:
    print('No record sets found in metadata. Will attempt to load default record set.')
# Attempt to enumerate available record sets
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']} | Name: {rs.get('name', 'Unnamed')}")

# Try loading sample records for each record set
loaded_record_set_ids = []
if record_sets:
    for rs in record_sets:
        print(f"\nSample records from RecordSet @id: {rs['@id']}")
        for rec in dataset.records(record_set=rs['@id']):
            print(rec)
            loaded_record_set_ids.append(rs['@id'])
            break  # Only show one example per record set


## 3. Data Extraction
Load data from specific record set(s) into pandas DataFrames for analysis.

All entities are referenced by their `@id`, as per Croissant schema conventions.

In [ ]:
# If no record sets found from metadata, attempt to use known record set ids or default
if not loaded_record_set_ids:
    # Try with default or guessed record set id
    # Replace with actual record set ids if known
    candidate_record_sets = ['cr:RecordSet', 'dv:ExperimentDataset']
else:
    candidate_record_sets = loaded_record_set_ids

dataframes = {}
for record_set_id in candidate_record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame columns for RecordSet @id '{record_set_id}':")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load records for RecordSet '{record_set_id}': {e}")
        continue

# Display the available DataFrames and their keys
print(f"\nLoaded DataFrames for record sets: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All field references are made via their `@id`.

In [ ]:
# Assuming the main data is loaded into the first record set
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using RecordSet @id: {record_set_id} for EDA.")

    # Show column names for manual field selection
    print("Available columns:")
    print(df.columns.tolist())

    # Try to find numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    print("Numeric columns:")
    print(numeric_cols)

    # If no numeric column found, try to infer from standard names
    if len(numeric_cols) == 0:
        for col in df.columns:
            if 'age' in col.lower() or 'interval' in col.lower():
                numeric_field = col
                break
        else:
            numeric_field = df.columns[0]  # fallback
    else:
        numeric_field = numeric_cols[0]

    print(f"\nSelected numeric field for analysis: {numeric_field}")

    threshold = 10
    try:
        filtered_df = df[df[numeric_field] > threshold]
    except Exception:
        # Attempt to coerce to numeric
        filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
    print(f"Filtered records with '{numeric_field}' > {threshold}:")
    print(filtered_df.head())

    # Normalization
    try:
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    except Exception:
        filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    categorical_cols = [col for col in df.columns if df[col].dtype == 'object' or df[col].dtype.name == 'category']
    if categorical_cols:
        group_field = categorical_cols[0]
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())
else:
    print("No dataframes loaded for EDA. Please check record set IDs/metadata.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize numeric field distribution and relationship to grouping field
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    numeric_field = None
    # Try to find a numeric or interpretable column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try with 'age' or 'interval' fields
        for col in df.columns:
            if 'age' in col.lower() or 'interval' in col.lower():
                numeric_field = col
                break
    if numeric_field:
        plt.figure(figsize=(8,4))
        sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce'), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()

    # Try to plot grouping field vs numeric
    categorical_cols = [col for col in df.columns if df[col].dtype == 'object']
    if categorical_cols and numeric_field:
        group_field = categorical_cols[0]
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field], y=pd.to_numeric(df[numeric_field], errors='coerce'))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("No dataframe to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to:
- Load Croissant metadata and records using the `mlcroissant` library.
- Identify available record sets and reference them via their `@id`.
- Extract records into pandas DataFrames for further analysis.
- Perform basic exploratory data analysis: filtering, normalization, and grouping.
- Visualize numeric and categorical data relationships.

The FAIR^2 dataset contains rich clinicopathological and molecular data for survivors with second primary colorectal cancer. Analysis can be extended to stratify by biomarkers (e.g. MSI-H status) and anatomical site distributions, supporting further research and clinical insights.